In [ ]:
!pip install librosa
!pip install torch
!pip install torchaudio
!pip install scipy
!pip install tqdm
!pip install pypinyin
!pip install torch
!pip install numpy

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
import os, random, shutil, glob, subprocess
import numpy as np
import librosa
import torchaudio
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset
from scipy.signal import butter, filtfilt
from collections import defaultdict
from tqdm import tqdm
from pypinyin import pinyin, Style
# Add these imports to your existing imports cell
import matplotlib.pyplot as plt
import seaborn as sns
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
seed = 67
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
#from google.colab import files
#uploaded = files.upload()

In [ ]:
#!ls /content/drive
#!ls /content/drive/MyDrive
#!ls /content/drive/MyDrive/Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!tar -xvzf "/content/drive/MyDrive/data_aishell.tgz" -C /content

Mounted at /content/drive
data_aishell/
data_aishell/wav/
data_aishell/wav/S0724.tar.gz
data_aishell/wav/S0725.tar.gz
data_aishell/wav/S0726.tar.gz
data_aishell/wav/S0727.tar.gz
data_aishell/wav/S0728.tar.gz
data_aishell/wav/S0729.tar.gz
data_aishell/wav/S0730.tar.gz
data_aishell/wav/S0731.tar.gz
data_aishell/wav/S0732.tar.gz
data_aishell/wav/S0733.tar.gz
data_aishell/wav/S0734.tar.gz
data_aishell/wav/S0735.tar.gz
data_aishell/wav/S0736.tar.gz
data_aishell/wav/S0737.tar.gz
data_aishell/wav/S0738.tar.gz
data_aishell/wav/S0739.tar.gz
data_aishell/wav/S0740.tar.gz
data_aishell/wav/S0741.tar.gz
data_aishell/wav/S0742.tar.gz
data_aishell/wav/S0743.tar.gz
data_aishell/wav/S0744.tar.gz
data_aishell/wav/S0745.tar.gz
data_aishell/wav/S0746.tar.gz
data_aishell/wav/S0747.tar.gz
data_aishell/wav/S0748.tar.gz
data_aishell/wav/S0749.tar.gz
data_aishell/wav/S0750.tar.gz
data_aishell/wav/S0751.tar.gz
data_aishell/wav/S0752.tar.gz
data_aishell/wav/S0753.tar.gz
data_aishell/wav/S0754.tar.gz
data_aishell

In [ ]:
import glob
import subprocess
import os

wav_root = "/content/data_aishell/wav"
archives = glob.glob(os.path.join(wav_root, "*.tar.gz"))
print(f"Found {len(archives)} archives in wav_root")

for archive in archives:
    print("Extracting", archive)
    subprocess.run(["tar", "-xvzf", archive, "-C", wav_root])

Found 400 archives in wav_root
Extracting /content/data_aishell/wav/S0128.tar.gz
Extracting /content/data_aishell/wav/S0334.tar.gz
Extracting /content/data_aishell/wav/S0054.tar.gz
Extracting /content/data_aishell/wav/S0095.tar.gz
Extracting /content/data_aishell/wav/S0755.tar.gz
Extracting /content/data_aishell/wav/S0916.tar.gz
Extracting /content/data_aishell/wav/S0194.tar.gz
Extracting /content/data_aishell/wav/S0152.tar.gz
Extracting /content/data_aishell/wav/S0035.tar.gz
Extracting /content/data_aishell/wav/S0003.tar.gz
Extracting /content/data_aishell/wav/S0360.tar.gz
Extracting /content/data_aishell/wav/S0212.tar.gz
Extracting /content/data_aishell/wav/S0217.tar.gz
Extracting /content/data_aishell/wav/S0340.tar.gz
Extracting /content/data_aishell/wav/S0146.tar.gz
Extracting /content/data_aishell/wav/S0731.tar.gz
Extracting /content/data_aishell/wav/S0171.tar.gz
Extracting /content/data_aishell/wav/S0714.tar.gz
Extracting /content/data_aishell/wav/S0248.tar.gz
Extracting /content

In [ ]:
!ls -lh /content/data_aishell/wav

total 15G
drwxr-xr-x  42 root  root 4.0K Dec  5 11:03 dev
-rw-r--r--   1 61083 fax   35M Jun 14  2017 S0002.tar.gz
-rw-r--r--   1 61083 fax   41M Jun 14  2017 S0003.tar.gz
-rw-r--r--   1 61083 fax   25M Jun 14  2017 S0004.tar.gz
-rw-r--r--   1 61083 fax   27M Jun 14  2017 S0005.tar.gz
-rw-r--r--   1 61083 fax   32M Jun 14  2017 S0006.tar.gz
-rw-r--r--   1 61083 fax   42M Jun 14  2017 S0007.tar.gz
-rw-r--r--   1 61083 fax   35M Jun 14  2017 S0008.tar.gz
-rw-r--r--   1 61083 fax   40M Jun 14  2017 S0009.tar.gz
-rw-r--r--   1 61083 fax   36M Jun 14  2017 S0010.tar.gz
-rw-r--r--   1 61083 fax   34M Jun 14  2017 S0011.tar.gz
-rw-r--r--   1 61083 fax   43M Jun 14  2017 S0012.tar.gz
-rw-r--r--   1 61083 fax   43M Jun 14  2017 S0013.tar.gz
-rw-r--r--   1 61083 fax   53M Jun 14  2017 S0014.tar.gz
-rw-r--r--   1 61083 fax   41M Jun 14  2017 S0015.tar.gz
-rw-r--r--   1 61083 fax   47M Jun 14  2017 S0016.tar.gz
-rw-r--r--   1 61083 fax   35M Jun 14  2017 S0017.tar.gz
-rw-r--r--   1 61083 fax   33M

In [ ]:
import os

wav_root = "/content/data_aishell/wav"

# Go inside the test split
test_path = os.path.join(wav_root, "test")

# Now list speaker folders
speaker_folders = [d for d in os.listdir(test_path)
                   if os.path.isdir(os.path.join(test_path, d))]

print("Speakers in test split:", speaker_folders[:5])

# Pick first speaker folder
first_speaker = speaker_folders[0]
speaker_path = os.path.join(test_path, first_speaker)

# List wavs
wav_files = [f for f in os.listdir(speaker_path) if f.endswith(".wav")]
print("Number of wav files:", len(wav_files))
print("Examples:", wav_files[:5])

Speakers in test split: ['S0764', 'S0912', 'S0768', 'S0767', 'S0908']
Number of wav files: 370
Examples: ['BAC009S0764W0399.wav', 'BAC009S0764W0263.wav', 'BAC009S0764W0342.wav', 'BAC009S0764W0325.wav', 'BAC009S0764W0166.wav']


In [ ]:
#per speaker
#train_folder = "aishell-1/train"
#subset_folder = "aishell-1/train_subset_50"
#os.makedirs(subset_folder, exist_ok=True)

#speaker_files = defaultdict(list)
#for root, _, files in os.walk(train_folder):
#    for f in files:
#        if f.endswith(".wav"):
#            speaker = os.path.basename(root)  # assumes speaker folder
#            speaker_files[speaker].append(os.path.join(root, f))

#subset_files = []
#random.seed(67)
#for speaker, files in speaker_files.items():
#    n_sample = max(1, int(len(files) * 0.5))  # 50%
#    subset_files.extend(random.sample(files, n_sample))

#for f in subset_files:
#    rel_path = os.path.relpath(f, train_folder)
#    dst = os.path.join(subset_folder, rel_path)
#    os.makedirs(os.path.dirname(dst), exist_ok=True)
#    shutil.copy2(f, dst)

In [ ]:
def lowpass_filter(data, cutoff_freq, sr, order = 4):
  nyquist = 0.5 * sr
  cutoff = cutoff_freq /nyquist
  b, a = butter(order, cutoff, btype = 'low', analog = False)
  return filtfilt (b, a, data)

In [ ]:
!pip install datasets
!pip install accelerate -U
!pip install numpy
!pip install pandas

In [ ]:
def add_gaussian_noise(y, snr_db):
    sig_power = np.mean(y**2)
    snr_linear = 10 ** (snr_db / 10)
    noise_power = sig_power / snr_linear
    noise = np.random.normal(0, np.sqrt(noise_power), size=y.shape)

    return y + noise

In [ ]:
def extract_features_improved(wav_path, sr=16000, apply_noise=False):
    """
    IMPROVED: Better features with delta and normalization
    """
    y, sr = librosa.load(wav_path, sr=sr)

    if apply_noise:
        # Variable SNR for better augmentation
        snr_db = random.uniform(15, 25)
        sig_power = np.mean(y**2)
        snr_linear = 10 ** (snr_db / 10)
        noise_power = sig_power / snr_linear
        noise = np.random.normal(0, np.sqrt(noise_power), size=y.shape)
        y = y + noise

    # 1. F0 with better thresholding
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, fmin=75, fmax=400)

    f0_raw = []
    for p, m in zip(pitches.T, magnitudes.T):
        valid_idx = np.where(m > np.median(m) * 0.3)[0]
        if len(valid_idx) > 0:
            f0_raw.append(p[valid_idx[0]])
        else:
            f0_raw.append(0)

    f0_raw = np.array(f0_raw)

    # Smooth F0
    try:
        f0 = lowpass_filter(f0_raw, cutoff_freq=20, sr=sr)
    except:
        f0 = f0_raw

    # 2. Add F0 delta (important for tone contours!)
    if len(f0) > 1:
        f0_delta = np.gradient(f0)
    else:
        f0_delta = np.zeros_like(f0)

    # 3. Energy features
    rms = librosa.feature.rms(y=y, hop_length=512)[0]

    # 4. MFCC with delta features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=512)
    mfcc_delta = librosa.feature.delta(mfcc)

    # 5. Zero Crossing Rate (voice activity)
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]

    # Align all features
    min_len = min(len(f0), len(f0_delta), len(rms), mfcc.shape[1], len(zcr))

    # Truncate
    f0 = f0[:min_len]
    f0_delta = f0_delta[:min_len]
    rms = rms[:min_len]
    zcr = zcr[:min_len]
    mfcc = mfcc[:, :min_len]
    mfcc_delta = mfcc_delta[:, :min_len]

    # Stack features: 1 + 1 + 1 + 1 + 13 + 13 = 30 features
    features = np.vstack([
        f0,            # 1
        f0_delta,      # 1
        rms,           # 1
        zcr,           # 1
        mfcc,          # 13
        mfcc_delta     # 13
    ]).T

    # Normalize features (important!)
    features = (features - np.mean(features, axis=0)) / (np.std(features, axis=0) + 1e-8)

    return features

In [ ]:

#Hann = librosa.stft(y, window='hann', n_fft = 2048, hop_length = 512) # windowing in case needed

#def extract_features(wav_path, sr=16000, apply_noise = False):
#  y, sr = librosa.load (wav_path, sr = sr)

 # if apply_noise:
 #   y = add_gaussian_noise (y,snr_db = 20)

#  pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
#  f0_raw = [p[np.argmax(m)] if np.any(m) else 0 for p, m in zip(pitches.T, magnitudes.T)]
# f0 = lowpass_filter(f0_raw, cutoff_freq = 20, sr = sr)

#  rms = librosa.feature.rms(y=y)[0]
#  mfcc = librosa.feature.mfcc (y=y, sr=sr, n_mfcc = 40)

  #shape aligning
#  min_len = min(len(f0), len(rms), mfcc.shape[1])
#  f0 = f0[:min_len]
#  rms = rms[:min_len]
#`  mfcc = mfcc [:, :min_len]

#  features = np.vstack([f0, rms, mfcc]).T
#  return features


In [ ]:
def extract_tones_fixed(text, n_frames=None):
    """
    FIXED: Better tone alignment with realistic syllable durations
    """
    tones = pinyin(text, style=Style.TONE3)
    tone_numbers = []

    for syl in tones:
        s = syl[0]
        if s[-1].isdigit():
            tone_numbers.append(int(s[-1]))
        else:
            tone_numbers.append(0)

    if n_frames is None:
        return np.array(tone_numbers)

    n_syllables = len(tone_numbers)

    if n_syllables == 0:
        return np.zeros(n_frames, dtype=int)


    # APPROACH: Proportional allocation with silence
    # Average Chinese syllable: ~250ms = ~8 frames at 31.25 Hz (16000/512)
    # But let's use more realistic mapping

    y_aligned = []

    # Method 1: Simple proportional (better than repeating)
    frames_per_syllable = max(10, n_frames // (n_syllables * 2))

    for i, tone in enumerate(tone_numbers):
        # Syllable frames
        syllable_frames = frames_per_syllable + random.randint(-2, 2)
        y_aligned.extend([tone] * syllable_frames)

        # Silence/transition (tone 0)
        if i < n_syllables - 1:  # Not after last syllable
            silence_frames = frames_per_syllable // 2
            y_aligned.extend([0] * silence_frames)

    # Adjust length
    if len(y_aligned) > n_frames:
        y_aligned = y_aligned[:n_frames]
    else:
        # Pad with silence
        y_aligned.extend([0] * (n_frames - len(y_aligned)))


    return np.array(y_aligned)

class AISHELLToneDataset(Dataset):
    def __init__(self, wav_root, transcript_path, apply_noise=False, subset_ratio=0.5):
        self.transcript = {}
        with open(transcript_path, "r", encoding="utf-8") as f:
            for line in f:
                utt_id, text = line.strip().split(" ", 1)
                self.transcript[utt_id] = text
        self.items = []
        for speaker in os.listdir(wav_root):
            spk_dir = os.path.join(wav_root, speaker)
            if not os.path.isdir(spk_dir):
                continue
            files = [f for f in os.listdir(spk_dir) if f.endswith(".wav") and f.replace(".wav", "") in self.transcript]
            if len(files) == 0:
                continue
            n_sample = max(1, min(len(files), int(len(files) * subset_ratio)))
            sampled_files = random.sample(files, n_sample)
            for wav_file in sampled_files:
                utt_id = wav_file.replace(".wav", "")
                self.items.append({
                    "wav_path": os.path.join(spk_dir, wav_file),
                    "text": self.transcript[utt_id],
                    "speaker": speaker
                })
        self.apply_noise = apply_noise
        print(f"Loaded {len(self.items)} utterances from AISHELL.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        features = extract_features_improved(item["wav_path"], apply_noise=self.apply_noise)
        tones = extract_tones_fixed(item["text"], n_frames=features.shape[0])
        X = torch.tensor(features, dtype=torch.float32)
        y = torch.tensor(tones, dtype=torch.long)
        lengths = torch.tensor(X.shape[0], dtype=torch.long)
        return X, y, lengths

def collate_fn(batch):
    X_list, y_list, lengths_list = zip(*batch)
    lengths_list, X_list, y_list = zip(*sorted(zip(lengths_list, X_list, y_list), key=lambda x: x[0], reverse=True))
    X_padded = torch.nn.utils.rnn.pad_sequence(X_list, batch_first=True)
    y_padded = torch.nn.utils.rnn.pad_sequence(y_list, batch_first=True, padding_value=-100)
    lengths = torch.stack(lengths_list)
    return X_padded, y_padded, lengths

In [ ]:
class ToneClassifier(nn.Module):
    def __init__(self, input_size=30, hidden_size=128, num_layers=2, num_classes=5, use_bilstm=True):
        super().__init__()
        self.use_bilstm = use_bilstm
        if use_bilstm:
            self.rnn = nn.LSTM(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=True
            )
            self.fc = nn.Linear(hidden_size*2, num_classes)
        else:
            self.rnn = nn.GRU(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True
            )
            self.fc = nn.Linear(hidden_size, num_classes)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, lengths):
        # Pack sequence
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
import torch.amp

model = ToneClassifier(input_size=30, hidden_size=128, num_layers=2, num_classes=5).to(device)
optimizer = optim.AdamW(model.parameters(), lr=4e-3, weight_decay = 1e-5)

def init_weights_improved(m):
    """
    BETTER initialization for RNNs
    """
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LSTM):
        for name, param in m.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)  # Better for RNNs!
            elif 'bias' in name:
                nn.init.zeros_(param.data)
                # Set forget gate bias to 1 for LSTM (helps memory)
                n = param.size(0)
                param.data[n//4:n//2].fill_(1.0)
    elif isinstance(m, nn.GRU):
        for name, param in m.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.zeros_(param.data)

model.apply(init_weights_improved)  # Use new initialization


# Replace your train_epoch function with this:
def train_epoch(model, dataloader, optimizer, scaler):
    model.train()
    total_loss = 0
    total_samples = 0

    loop = tqdm(dataloader, desc="Training")
    for X, y, lengths in loop:
        X, y, lengths = X.to(device), y.to(device), lengths.to(device)

        optimizer.zero_grad()

        # Forward pass with mixed precision
        with torch.amp.autocast('cuda'):  # or 'cuda' if using GPU
            outputs = model(X, lengths)

            # Reshape for loss calculation
            outputs_flat = outputs.reshape(-1, outputs.shape[-1])
            y_flat = y.reshape(-1)

            # Calculate loss (ignore padding)
            loss = F.cross_entropy(outputs_flat, y_flat, ignore_index=-100)

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / total_samples
    print(f"Epoch Training Loss: {avg_loss:.4f}")
    return avg_loss

def evaluate_complete(model, dataloader):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y, lengths in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X, lengths)
            preds = outputs.argmax(dim=-1)

            # Flatten and remove padding
            mask = y != -100
            all_preds.extend(preds[mask].cpu().numpy())
            all_targets.extend(y[mask].cpu().numpy())

    # Calculate metrics
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

    accuracy = accuracy_score(all_targets, all_preds)

    print(f"Overall Accuracy: {accuracy:.4f}")
    print("\nClassification Report (Tones 0-4):")
    print(classification_report(all_targets, all_preds,
                                target_names=['Silence', 'Tone1', 'Tone2', 'Tone3', 'Tone4']))

    # Calculate accuracy excluding silence (tone 0)
    tone_mask = np.array(all_targets) != 0
    if tone_mask.any():
        tone_accuracy = accuracy_score(
            np.array(all_targets)[tone_mask],
            np.array(all_preds)[tone_mask]
        )
        print(f"Accuracy on actual tones (1-4): {tone_accuracy:.4f}")

    return accuracy

In [ ]:
train_dataset = AISHELLToneDataset(
    wav_root="/content/data_aishell/wav/train",
    transcript_path="/content/data_aishell/transcript/aishell_transcript_v0.8.txt",
    apply_noise=True,
    subset_ratio=0.25
)

val_dataset = AISHELLToneDataset(
    wav_root="/content/data_aishell/wav/dev",
    transcript_path="/content/data_aishell/transcript/aishell_transcript_v0.8.txt",
    apply_noise=False,
    subset_ratio=0.25
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

Loaded 29894 utterances from AISHELL.
Loaded 3565 utterances from AISHELL.


In [ ]:
# Initialize with fixed code
scaler = torch.amp.GradScaler()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

# Quick training (2-3 epochs should be enough for assignment)
num_epochs = 3
best_acc = 0

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")

    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scaler)

    # Validate
    val_acc = evaluate_complete(model, val_loader)

    # Learning rate scheduling
    scheduler.step(train_loss)

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_tone_model.pth')
        print(f"Saved best model with accuracy: {val_acc:.4f}")


Epoch 1/3


Training: 100%|██████████| 7474/7474 [23:03<00:00,  5.40it/s, loss=1.08]


Epoch Training Loss: 1.2047
Overall Accuracy: 0.5511

Classification Report (Tones 0-4):


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

     Silence       0.59      0.90      0.71    311950
       Tone1       0.00      0.00      0.00     46057
       Tone2       0.00      0.00      0.00     43787
       Tone3       0.00      0.00      0.00     35687
       Tone4       0.00      0.00      0.00     72207

    accuracy                           0.55    509688
   macro avg       0.12      0.18      0.14    509688
weighted avg       0.36      0.55      0.43    509688

Accuracy on actual tones (1-4): 0.0000
Saved best model with accuracy: 0.5511

Epoch 2/3


Training:  15%|█▍        | 1109/7474 [03:22<19:21,  5.48it/s, loss=1.19]


KeyboardInterrupt: 

In [ ]:
# ======================
# DIAGNOSTIC SCRIPT
# ======================
print("\n=== DIAGNOSTICS ===")

# 1. Check model output distribution
model.eval()
sample_X, sample_y, sample_len = next(iter(val_loader))
sample_X = sample_X[:1].to(device)
sample_len = sample_len[:1].to(device)

with torch.no_grad():
    output = model(sample_X, sample_len)
    probs = F.softmax(output, dim=-1)
    print(f"Output shape: {output.shape}")
    print(f"Average probability per class: {probs.mean(dim=[0,1])}")
    print(f"Max probability: {probs.max().item():.3f}, Min: {probs.min().item():.3f}")

# 2. Check if predictions are random
preds = output.argmax(dim=-1)
unique, counts = torch.unique(preds, return_counts=True)
print(f"Prediction distribution: {dict(zip(unique.tolist(), counts.tolist()))}")

# 3. Check gradient norms
model.train()
X, y, lengths = next(iter(train_loader))
X, y, lengths = X.to(device), y.to(device), lengths.to(device)

optimizer.zero_grad()
output = model(X, lengths)
loss = F.cross_entropy(output.view(-1, output.shape[-1]),
                       y.view(-1), ignore_index=-100)
loss.backward()

total_grad_norm = 0
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        total_grad_norm += grad_norm ** 2
        if grad_norm > 0:
            print(f"{name}: grad norm = {grad_norm:.6f}")

print(f"Total gradient norm: {total_grad_norm**0.5:.6f}")
print("="*30)


=== DIAGNOSTICS ===
Output shape: torch.Size([1, 227, 5])
Average probability per class: tensor([0.5701, 0.1000, 0.0930, 0.0759, 0.1610], device='cuda:0')
Max probability: 0.978, Min: 0.000
Prediction distribution: {0: 218, 4: 9}
rnn.weight_ih_l0: grad norm = 0.011065
rnn.weight_hh_l0: grad norm = 0.009933
rnn.bias_ih_l0: grad norm = 0.001925
rnn.bias_hh_l0: grad norm = 0.001925
rnn.weight_ih_l0_reverse: grad norm = 0.017025
rnn.weight_hh_l0_reverse: grad norm = 0.008863
rnn.bias_ih_l0_reverse: grad norm = 0.001878
rnn.bias_hh_l0_reverse: grad norm = 0.001878
rnn.weight_ih_l1: grad norm = 0.016641
rnn.weight_hh_l1: grad norm = 0.002683
rnn.bias_ih_l1: grad norm = 0.002390
rnn.bias_hh_l1: grad norm = 0.002390
rnn.weight_ih_l1_reverse: grad norm = 0.021267
rnn.weight_hh_l1_reverse: grad norm = 0.001531
rnn.bias_ih_l1_reverse: grad norm = 0.003212
rnn.bias_hh_l1_reverse: grad norm = 0.003212
fc.weight: grad norm = 0.093044
fc.bias: grad norm = 0.068704
Total gradient norm: 0.121459


In [ ]:
#num_epochs = 3
#for epoch in range(num_epochs):
#    print(f"Epoch {epoch+1}/{num_epochs}")
#    train_epoch(model, train_loader)
#    evaluate(model, val_loader)

In [ ]:
# ============================================
# TEST FEATURE DIMENSION
# ============================================

print("Testing feature extraction...")

# Get first sample from dataset
sample_item = train_dataset[0]
X_sample, y_sample, length_sample = sample_item

print(f"Sample feature shape: {X_sample.shape}")
print(f"Input size should be: {X_sample.shape[1]}")

# Also test with a random audio file
test_wav = train_dataset.items[0]["wav_path"]
features = extract_features_improved(test_wav)
print(f"\nDirect feature extraction shape: {features.shape}")
print(f"Number of features: {features.shape[1]}")

# Check what features we have
if features.shape[1] == 30:
    print("✓ Confirmed: 30 features (f0, f0_delta, rms, zcr, 13 MFCCs, 13 delta-MFCCs)")
elif features.shape[1] == 29:
    print("✓ Confirmed: 29 features")
else:
    print(f"⚠ Unexpected: {features.shape[1]} features")

Testing feature extraction...
Sample feature shape: torch.Size([110, 30])
Input size should be: 30

Direct feature extraction shape: (110, 30)
Number of features: 30
✓ Confirmed: 30 features (f0, f0_delta, rms, zcr, 13 MFCCs, 13 delta-MFCCs)


In [ ]:
# ============================================
# PARAMETER OPTIMIZATION FIX
# ============================================

print("OPTIMIZING PARAMETERS FOR IMBALANCED DATA")
print("="*50)

# 1. HARD RESET with better initialization
import torch.nn.init as init

class BetterToneClassifier(nn.Module):
    def __init__(self, input_size=30, hidden_size=256, num_layers=2, num_classes=5, dropout=0.5):
        super().__init__()

        # Larger BiLSTM
        self.rnn = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Attention layer (simple)
        self.attention = nn.Linear(hidden_size * 2, 1)

        # Classifier
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        self.dropout = nn.Dropout(dropout)

        # Initialize properly for imbalanced data
        self._init_weights()

    def _init_weights(self):
        # Initialize LSTM weights with larger variance
        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                if 'ih' in name:
                    init.xavier_uniform_(param)
                elif 'hh' in name:
                    init.orthogonal_(param)
            elif 'bias' in name:
                init.zeros_(param)
                # LSTM forget gate bias trick
                n = param.size(0)
                param.data[n//4:n//2].fill_(1.0)

        # Initialize final layer to not favor any class
        init.xavier_uniform_(self.fc.weight)
        init.constant_(self.fc.bias, 0.0)
        init.xavier_uniform_(self.attention.weight)
        init.constant_(self.attention.bias, 0.0)

    def forward(self, x, lengths):
        # Pack sequence
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=True
        )

        # RNN
        packed_out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        # Simple attention
        attention_weights = torch.softmax(self.attention(out).squeeze(-1), dim=-1)
        attention_weights = attention_weights.unsqueeze(-1)
        context = torch.sum(out * attention_weights, dim=1)

        # Classification
        out = self.dropout(context)
        out = self.fc(out)

        # Return frame-level and utterance-level predictions
        return out.unsqueeze(1).repeat(1, x.size(1), 1)  # Broadcast to sequence length

# 2. Create model with better parameters
model = BetterToneClassifier(
    input_size=30,      # Confirmed from your test
    hidden_size=256,    # Increased from 128
    num_layers=2,       # Same
    num_classes=5,      # Same
    dropout=0.5         # Increased regularization
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# 3. AGGRESSIVE optimizer settings
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,            # Higher learning rate
    weight_decay=1e-3,  # More regularization
    betas=(0.9, 0.999)
)

# 4. FOCAL LOSS with extreme class weights
print("\nCalculating extreme class weights...")

# Get actual distribution from first few batches
tone_counts = {0:0, 1:0, 2:0, 3:0, 4:0}
for X, y, lengths in train_loader:
    y_flat = y.view(-1)
    mask = y_flat != -100
    for tone in y_flat[mask]:
        tone_counts[tone.item()] += 1
    if sum(tone_counts.values()) > 100000:  # Enough samples
        break

total = sum(tone_counts.values())
print("Tone distribution (sample):")
for tone in range(5):
    print(f"  Tone {tone}: {tone_counts[tone]:,} ({tone_counts[tone]/total*100:.1f}%)")

# EXTREME weights - make rare tones 50x more important
class_weights = torch.tensor([
    (total / (tone_counts[i] + 1)) ** 0.5 for i in range(5)  # Square root for less extreme
]).float().to(device)

# Normalize
class_weights = class_weights / class_weights.sum() * 5
print(f"\nClass weights: {class_weights.tolist()}")
print(f"Tone 1 weight is {class_weights[1]/class_weights[0]:.1f}x Tone 0 weight")

# 5. Focal loss implementation
def focal_loss_with_weights(pred, target, weights, gamma=2.0, alpha=None):
    ce_loss = F.cross_entropy(pred, target, reduction='none', ignore_index=-100)

    # Apply class weights
    if weights is not None:
        weight_tensor = weights[target]
        ce_loss = ce_loss * weight_tensor

    # Focal loss component
    pt = torch.exp(-ce_loss)
    focal_loss = ((1 - pt) ** gamma) * ce_loss

    return focal_loss.mean()

# 6. Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=500,  # First restart after 500 steps
    T_mult=2,  # Double each time
    eta_min=1e-5
)

# 7. Test forward pass
print("\nTesting forward pass...")
model.eval()
with torch.no_grad():
    X, y, lengths = next(iter(train_loader))
    X, y, lengths = X.to(device), y.to(device), lengths.to(device)
    outputs = model(X, lengths)
    preds = outputs.argmax(dim=-1)

    print(f"Input shape: {X.shape}")
    print(f"Output shape: {outputs.shape}")

    # Check prediction distribution
    mask = y != -100
    pred_dist = torch.bincount(preds[mask].flatten(), minlength=5)
    print(f"\nInitial prediction distribution:")
    for tone in range(5):
        print(f"  Tone {tone}: {pred_dist[tone].item()} predictions")

print("\n" + "="*50)
print("READY FOR TRAINING!")
print("Parameters optimized for imbalanced tone classification")
print("="*50)

OPTIMIZING PARAMETERS FOR IMBALANCED DATA
Model parameters: 2,169,862

Calculating extreme class weights...
Tone distribution (sample):
  Tone 0: 57,373 (57.2%)
  Tone 1: 10,033 (10.0%)
  Tone 2: 9,686 (9.7%)
  Tone 3: 7,732 (7.7%)
  Tone 4: 15,519 (15.5%)

Class weights: [0.47748851776123047, 1.1417827606201172, 1.162052869796753, 1.3006081581115723, 0.9180676341056824]
Tone 1 weight is 2.4x Tone 0 weight

Testing forward pass...
Input shape: torch.Size([4, 211, 30])
Output shape: torch.Size([4, 211, 5])

Initial prediction distribution:
  Tone 0: 128 predictions
  Tone 1: 0 predictions
  Tone 2: 318 predictions
  Tone 3: 0 predictions
  Tone 4: 96 predictions

READY FOR TRAINING!
Parameters optimized for imbalanced tone classification


In [ ]:
# ============================================
# TRAINING LOOP FOR NEW MODEL
# ============================================

import torch
import torch.nn.functional as F

EPOCHS = 5
max_grad_norm = 5.0

def train_one_epoch(model, train_loader, optimizer, scheduler, class_weights):
    model.train()
    total_loss = 0
    total_frames = 0

    for X, y, lengths in train_loader:
        X, y, lengths = X.to(device), y.to(device), lengths.to(device)

        optimizer.zero_grad()

        # Forward
        outputs = model(X, lengths)             # [B, T, 5]
        outputs = outputs.view(-1, 5)
        y_flat = y.view(-1)

        # Compute loss
        loss = focal_loss_with_weights(
            outputs,
            y_flat,
            class_weights,
            gamma=2.0
        )

        # Backprop
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * (y_flat != -100).sum().item()
        total_frames += (y_flat != -100).sum().item()

    return total_loss / total_frames


def evaluate(model, val_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y, lengths in val_loader:
            X, y, lengths = X.to(device), y.to(device), lengths.to(device)

            outputs = model(X, lengths)      # [B, T, 5]
            preds = outputs.argmax(-1)

            mask = (y != -100)
            correct += (preds[mask] == y[mask]).sum().item()
            total += mask.sum().item()

    return correct / total if total > 0 else 0

In [ ]:
best_val_acc = 0.0

print("\n============= TRAINING START =============")

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        class_weights
    )

    val_acc = evaluate(model, val_loader)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Acc:    {val_acc*100:.2f}%")

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_tone_model.pt")
        print("  ✔ Saved new best model!")

print("\n============= TRAINING COMPLETE =============")
print(f"Best validation accuracy: {best_val_acc*100:.2f}%")


============= TRAINING START =============


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
